# 🗓️ 12일차 스터디 노트북 — 큐 (Queue) & 링 버퍼

**오늘 범위**: 04-2 큐란? → FIFO → 인큐/디큐/피크 → front/rear/no → **링 버퍼(ring buffer)** → FixedQueue 클래스 → 덱(deque) → 링 버퍼 활용(마지막 n개 저장)

*(04장 스택과 큐 완주!)*

## 난이도 태그
- 🟢 **기본** — 전원 필수 (개념을 쓸 수 있는지)
- 🟡 **표준** — 팀 목표선 (왜 그런지 + 변형/디버깅)
- 🔴 **심화** — 도전 (효율·엣지케이스·일반화)

## 유형 태그
[예측] 실행 전 결과 맞히기 · [구현] TODO 채우기 · [디버깅] 버그 찾기 · [설명] 왜인지 서술 · [실험] 직접 찍어보기

## 푸는 법
1. 위에서부터 순서대로 (Remind → 본문 🟢 → 🟡 → 🔴)
2. 코드 셀은 직접 실행해서 확인
3. 막히면 힌트만 보고, 정답은 맨 아래에서
4. 금요일 코딩테스트는 🟢🟡 범위

> 📁 아래 "부록: FixedQueue 전체 코드" 셀을 **먼저 실행**하면 `fixed_queue.py` 없이도 노트북이 단독으로 돌아가.

> ⚠️ **오늘의 하이라이트**: 큐를 배열로 순진하게 만들면 디큐가 O(n)이야. **링 버퍼**가 그걸 O(1)로 바꾸는 마법이 오늘의 핵심. 6번 문제에서 직접 비교해봐.

---

## 🔁 [Remind] 워밍업 — 11일차 스택 복습 (큐와 정반대!)

스택(LIFO)과 큐(FIFO)는 쌍둥이면서 정반대야. 비교하면서 시작하자.

### R-1. 🟢 [설명] 스택 vs 큐, 뭐가 반대야?

11일차 스택은 **LIFO**(후입선출), 오늘 큐는 **FIFO**(선입선출)야.

- 스택은 push/pop이 **한쪽 끝(꼭대기)**에서만 일어났어. 큐는 인큐/디큐가 각각 어디서 일어나?
- 스택은 포인터가 `ptr` **하나**였는데, 큐는 왜 `front`, `rear` **둘**이 필요할까?
- 은행 창구 줄서기가 큐의 비유인데, 스택을 일상 비유로 들면 뭐가 있을까?

*(여기에 답 작성)*

### R-2. 🟡 [디버깅] 스택 push/pop 순서 (11일차 회수)

11일차에 push는 "넣고→증가", pop은 "감소→반환"이라고 했지. 아래 pop은 순서가 틀렸어. 뭐가 문제일까?

In [ ]:
class BuggyStack:
    def __init__(self, cap=8):
        self.stk = [None]*cap; self.ptr = 0
    def push(self, v):
        self.stk[self.ptr] = v; self.ptr += 1
    def pop(self):
        x = self.stk[self.ptr]   # <- ptr 감소 전에 접근
        self.ptr -= 1
        return x

s = BuggyStack()
s.push(10); s.push(20); s.push(30)
print('pop:', s.pop())   # 30이 나와야 하는데?


---
## 📦 부록 — FixedQueue 전체 코드

`fixed_queue.py`가 같은 폴더에 없다면 **이 셀을 먼저 실행**해. (`count`, `__contains__`는 리뷰에서 지적한 대로 고친 버전)

In [ ]:
from typing import Any

class FixedQueue:
    class Empty(Exception):
        '''비어 있는 FixedQueue에서 디큐 또는 피크할 때 내보내는 예외'''
        pass
    class Full(Exception):
        '''가득 찬 FixedQueue에 인큐할 때 내보내는 예외'''
        pass

    def __init__(self, capacity: int) -> None:
        self.no = 0          # 현재 데이터 개수
        self.front = 0       # 맨 앞 원소 커서
        self.rear = 0        # 맨 끝 원소 커서 (다음 인큐 위치)
        self.capacity = capacity
        self.que = [None] * capacity

    def __len__(self) -> int:
        return self.no

    def is_empty(self) -> bool:
        return self.no <= 0

    def is_full(self) -> bool:
        return self.no >= self.capacity

    def enque(self, x: Any) -> None:
        if self.is_full():
            raise FixedQueue.Full
        self.que[self.rear] = x
        self.rear += 1
        self.no += 1
        if self.rear == self.capacity:   # 링 버퍼 순환
            self.rear = 0

    def deque(self) -> Any:
        if self.is_empty():
            raise FixedQueue.Empty
        x = self.que[self.front]
        self.front += 1
        self.no -= 1
        if self.front == self.capacity:  # 링 버퍼 순환
            self.front = 0
        return x

    def peek(self) -> Any:
        if self.is_empty():
            raise FixedQueue.Empty
        return self.que[self.front]

    def find(self, value: Any) -> Any:
        for i in range(self.no):
            idx = (i + self.front) % self.capacity
            if self.que[idx] == value:
                return idx
        return -1

    def count(self, value: Any) -> int:      # 리뷰 반영: bool -> int
        c = 0
        for i in range(self.no):
            idx = (i + self.front) % self.capacity
            if self.que[idx] == value:
                c += 1
        return c

    def __contains__(self, value: Any) -> bool:   # 리뷰 반영: > 0
        return self.count(value) > 0

    def clear(self) -> None:
        self.no = self.front = self.rear = 0

    def dump(self) -> None:
        if self.is_empty():
            print('큐가 비어 있습니다.')
        else:
            for i in range(self.no):
                print(self.que[(i + self.front) % self.capacity], end=' ')
            print()

print('FixedQueue 준비 완료')


---
## 📖 오늘의 핵심 개념

### 큐(queue) = FIFO
데이터를 임시 저장하는 자료구조. 입출력 순서는 **선입선출(FIFO, First In First Out)** — 가장 먼저 넣은 걸 가장 먼저 꺼냄. (스택 LIFO와 **정반대**)

- **인큐(enque)**: 데이터 넣기 (rear 쪽)
- **디큐(deque)**: 데이터 꺼내기 (front 쪽)
- **프런트(front)**: 맨 앞 (다음에 디큐될 쪽)
- **리어(rear)**: 맨 끝 (다음에 인큐될 쪽)

비유: 은행 창구 줄, 마트 계산 줄. 먼저 온 사람이 먼저 처리됨.

### 왜 배열로 순진하게 만들면 느릴까?
`que[0]`을 항상 맨 앞으로 고정하면: 디큐할 때마다 **뒤의 모든 원소를 한 칸씩 앞으로 당겨야** 해 → **O(n)**. 인큐는 O(1)인데 디큐가 O(n)이라 비효율.

### 링 버퍼(ring buffer) — 오늘의 핵심 🎯
배열 끝과 앞을 **논리적으로 연결**한 원형 구조. 원소를 옮기지 않고 **front/rear 커서만 이동**시켜서 인큐·디큐 모두 **O(1)**!

- **front**: 맨 앞 원소의 인덱스
- **rear**: 맨 끝 원소 **바로 뒤**의 인덱스 (다음 인큐될 자리)
- front/rear는 **논리적 순서**일 뿐, 배열의 물리적 순서가 아님

커서가 배열 끝(`capacity`)에 도달하면 `% capacity`로 **0으로 되돌아옴**. (8~9일차 해시의 `% capacity` 순환과 같은 원리!)

### 3개의 커서 (스택은 1개였는데!)
| 변수 | 의미 |
|---|---|
| `front` | 맨 앞 원소 인덱스 |
| `rear` | 맨 끝 원소 바로 뒤 인덱스 |
| `no` | 현재 데이터 개수 |

**왜 `no`가 따로 필요할까?** — `front == rear`일 때 큐가 **비었는지(no=0)** 가득 찼는지(no=capacity)** 구분할 수 없어! 두 상태 모두 front와 rear가 같아지거든. 그래서 개수 `no`를 따로 들고 판단해. (교재 173p 그림 4-10)

### enque / deque 의 커서 조작
```python
def enque(self, x):
    if self.is_full(): raise Full
    self.que[self.rear] = x       # rear 자리에 넣고
    self.rear += 1                # rear 전진
    self.no += 1
    if self.rear == self.capacity:  # 끝에 닿으면
        self.rear = 0               # 0으로 순환

def deque(self):
    if self.is_empty(): raise Empty
    x = self.que[self.front]      # front 자리에서 꺼내고
    self.front += 1               # front 전진
    self.no -= 1
    if self.front == self.capacity:
        self.front = 0
    return x
```

### find/count/dump 의 인덱스 변환
논리적 i번째 원소의 **물리적 인덱스** = `(i + front) % capacity`. 이 변환이 링 버퍼 순회의 핵심.

### 덱(deque, double-ended queue)
양쪽 끝에서 **모두** 삽입·삭제 가능한 자료구조. 큐 + 스택을 합친 형태. 파이썬 `collections.deque`. (11일차 stack.py에서 이미 썼던 것)

> ⚠️ 헷갈림 주의: 큐의 "디큐(dequeue, 꺼내기)"와 자료구조 "덱(deque)"은 발음이 비슷하지만 **다른 것**! 우리 코드의 메서드 이름 `deque()`는 "디큐(꺼내기)" 동작이야.

---
## 🟢 기본 문제

### 1. [예측] FIFO 손으로 추적하기

빈 `FixedQueue(4)`에 다음을 수행해. **실행 전에** 각 단계의 `front`, `rear`, `no`와 큐 내용을 적어봐.

```
enque(10) → enque(20) → enque(30) → deque() → enque(40) → enque(50)
```

| 연산 | front | rear | no | 큐 내용(논리적) | 반환 |
|---|---|---|---|---|---|
| enque(10) | ? | ? | ? | ? | - |
| enque(20) | ? | ? | ? | ? | - |
| enque(30) | ? | ? | ? | ? | - |
| deque() | ? | ? | ? | ? | ? |
| enque(40) | ? | ? | ? | ? | - |
| enque(50) | ? | ? | ? | ? | - |

**힌트**: capacity=4에서 rear가 어떻게 되는지, 특히 enque(50)에서 rear가 순환하는지 잘 봐.

In [ ]:
# 예측 먼저 하고 실행!
q = FixedQueue(4)
for cmd in ['e10', 'e20', 'e30', 'd', 'e40', 'e50']:
    if cmd.startswith('e'):
        q.enque(int(cmd[1:]))
        print(f'enque({cmd[1:]}) -> front={q.front}, rear={q.rear}, no={q.no}, dump=', end='')
        q.dump()
    else:
        x = q.deque()
        print(f'deque()={x} -> front={q.front}, rear={q.rear}, no={q.no}, dump=', end='')
        q.dump()


### 2. [구현] enque 채우기 (링 버퍼 순환)

TODO를 채워봐. 순환 처리가 핵심이야.

In [ ]:
class MyQueue:
    class Empty(Exception): pass
    class Full(Exception): pass

    def __init__(self, capacity):
        self.no = 0; self.front = 0; self.rear = 0
        self.capacity = capacity
        self.que = [None] * capacity

    def is_full(self): return self.no >= self.capacity

    def enque(self, x):
        if self.is_full():
            raise MyQueue.Full
        self.que[self.rear] = x
        # TODO: rear 전진, no 증가, 그리고 rear가 capacity에 닿으면 0으로 순환 (3줄)
        ___
        ___
        ___

    def dump(self):
        for i in range(self.no):
            print(self.que[(i + self.front) % self.capacity], end=' ')
        print()

q = MyQueue(3)
q.enque(1); q.enque(2); q.enque(3)
q.dump()          # 1 2 3
print(q.rear)     # 0 (3에 닿아서 순환했어야 함)


### 3. [설명] peek는 왜 front/rear/no를 안 바꿔?

11일차 스택의 peek처럼, 큐의 peek도 "맨 앞을 들여다보기만" 해.

- `deque`와 `peek`의 차이를 커서 관점에서 설명해봐.
- `peek`을 3번 연속 호출하면? `deque`를 3번 연속 호출하면?

*(여기에 답 작성)*

---
## 🟡 표준 문제

### 4. [설명] `no`가 없으면 무슨 일이 생기나 🔥

교재 173p: "변수 front와 rear의 값이 같을 경우 큐가 비어 있는지 또는 가득 차 있는지 구별하기 위해 `no`가 필요합니다."

아래 실험으로 **왜 front/rear만으론 부족한지** 직접 확인해봐.

In [ ]:
# no 없이 front == rear로만 빈/가득을 판단하려는 (실패하는) 시도
q = FixedQueue(4)

print('=== 빈 상태 ===')
print(f'front={q.front}, rear={q.rear}  ->  front==rear? {q.front == q.rear}')
print(f'실제로는 비어있음 (no={q.no})')
print()

print('=== 가득 채운 상태 ===')
for v in [1, 2, 3, 4]:
    q.enque(v)
print(f'front={q.front}, rear={q.rear}  ->  front==rear? {q.front == q.rear}')
print(f'실제로는 가득 참 (no={q.no})')
print()
print('결론: front==rear가 빈 상태에서도 True, 가득 찬 상태에서도 True!')
print('     -> 오직 no만이 두 상태를 구분할 수 있다.')


**(a)** 빈 상태와 가득 찬 상태에서 `front == rear`가 둘 다 참인 이유를 설명해봐.

**(b)** 만약 `no`를 안 쓰고 이 문제를 해결하려면 어떤 방법이 있을까? (힌트: 배열 한 칸을 일부러 비워두는 방법이 유명해)

*(여기에 답 작성)*

---
### 5. [디버깅] `__contains__` & `count` 타입 (11일차와 판박이) 🔥

내가 짠 `fixed_queue.py`에 이런 게 있었어 (11일차 stack.py와 **똑같은** 실수):

```python
def count(self, value: Any) -> bool:      # int를 반환하는데 bool이라 적음
    ...
    return c

def __contains__(self, value: Any) -> bool:
    return self.count(value)              # > 0 없이 int를 그대로
```

아래를 실행해서 확인해봐.

In [ ]:
class QueueBug:
    def __init__(self): self.data = [10, 20, 30, 10]
    def count(self, v): return self.data.count(v)   # int 반환
    def __contains__(self, v): return self.count(v)  # > 0 없음

q = QueueBug()
print('q.__contains__(10):', q.__contains__(10), '(True가 아니라...?)')
print('q.__contains__(99):', q.__contains__(99))
print('10 in q:', 10 in q, '| type:', type(10 in q))
print()
print('in 연산자는 bool로 강제변환하지만, __contains__ 직접 호출은 int를 뱉는다')


**(a)** 11일차 스택에서도 똑같은 걸 지적했었지. 이 두 줄을 어떻게 고쳐야 할까?

**(b)** `count`의 반환 타입이 `-> bool`인데 실제로는 개수(int)를 반환하는 게 왜 문제야? (8~9일차 어노테이션 논의와 연결)

*(여기에 답 작성)*

---
### 6. [실험] 순진한 큐 O(n) vs 링 버퍼 O(1) 🔥🔥

**오늘의 핵심.** 디큐할 때마다 원소를 앞으로 당기는 "순진한 큐"와, 커서만 옮기는 "링 버퍼 큐"의 연산 횟수를 직접 재봐.

In [ ]:
# 순진한 큐: 디큐할 때마다 앞으로 당김
class NaiveQueue:
    def __init__(self):
        self.data = []
        self.shift_count = 0   # 원소 이동 횟수 카운트
    def enque(self, x):
        self.data.append(x)
    def deque(self):
        x = self.data[0]
        # que[0]을 꺼내고 나머지를 앞으로 당김 (이게 O(n)!)
        for i in range(len(self.data) - 1):
            self.data[i] = self.data[i + 1]
            self.shift_count += 1
        self.data.pop()
        return x

# 링 버퍼 큐: 커서만 이동
class RingQueue:
    def __init__(self, cap):
        self.que = [None]*cap; self.front = 0; self.rear = 0; self.no = 0
        self.cap = cap
        self.move_count = 0    # 커서 이동 횟수
    def enque(self, x):
        self.que[self.rear] = x
        self.rear = (self.rear + 1) % self.cap
        self.no += 1
        self.move_count += 1
    def deque(self):
        x = self.que[self.front]
        self.front = (self.front + 1) % self.cap
        self.no -= 1
        self.move_count += 1
        return x

N = 1000
# 순진한 큐: N번 인큐 후 N번 디큐
nq = NaiveQueue()
for i in range(N): nq.enque(i)
for i in range(N): nq.deque()

# 링 버퍼 큐: 동일 작업
rq = RingQueue(N)
for i in range(N): rq.enque(i)
for i in range(N): rq.deque()

print(f'N = {N}')
print(f'순진한 큐 원소 이동 횟수: {nq.shift_count:,}  (~ N²/2)')
print(f'링 버퍼 커서 이동 횟수 : {rq.move_count:,}  (~ 2N)')
print(f'배율: 순진한 큐가 약 {nq.shift_count // rq.move_count}배 더 많은 연산')


**(a)** 순진한 큐의 이동 횟수가 왜 N²에 비례하는지 설명해봐. (힌트: 디큐 1번에 몇 개를 옮기지? 그걸 N번?)

**(b)** 링 버퍼는 왜 원소를 안 옮겨도 될까? 무엇을 옮기는 걸로 대체했지?

*(여기에 답 작성)*

---
## 🔴 심화 문제

### 7. [예측] 링 버퍼 활용 — 마지막 n개만 저장

`last_elements.py`의 핵심은 `a[cnt % n]`이야. 데이터를 계속 입력받아도 **배열엔 최근 n개만** 남아. 아래를 예측하고 실행해봐.

In [ ]:
def keep_last_n(values, n):
    '''values를 순서대로 넣되 최근 n개만 유지, 저장 순서대로 반환'''
    a = [None] * n
    cnt = 0
    for v in values:
        a[cnt % n] = v    # 링 버퍼: n으로 나눈 나머지 위치에 덮어씀
        cnt += 1
    # 저장된 것을 논리적 순서로 출력
    result = []
    start = cnt - n
    if start < 0:
        start = 0
    for i in range(start, cnt):
        result.append(a[i % n])
    return result

# n=10, 12개 입력 -> 앞 2개(15,17)는 버려지고 마지막 10개만
values = [15, 17, 64, 57, 99, 21, 0, 23, 44, 55, 97, 85]
print('입력:', values)
print('n=10 결과:', keep_last_n(values, 10))
print('  -> 15, 17이 버려지고 64부터 85까지 10개만 남아야 함')
print()
print('n=5 결과:', keep_last_n(values, 5))
print('  -> 마지막 5개만')


**(a)** 11번째 입력값(97)이 배열의 어느 인덱스에 저장되는지 `cnt % n`으로 계산해봐. 그 자리엔 원래 몇 번째 값이 있었지?

**(b)** 이게 오늘 배운 **링 버퍼**와 정확히 같은 원리인 이유를 설명해봐. (`% n`이 하는 역할은?)

*(여기에 답 작성)*

---
### 8. [구현] 큐로 최근 3개 이동평균 만들기

큐의 실전 응용 — **슬라이딩 윈도우**. 스트림으로 들어오는 숫자의 "최근 3개 평균"을 계속 구해봐. 큐가 가득 차면 가장 오래된 걸 디큐하고 새 걸 인큐하는 패턴이야.

In [ ]:
def moving_average(stream, window=3):
    '''각 시점에서 최근 window개의 평균을 리스트로 반환'''
    q = FixedQueue(window)
    result = []
    for x in stream:
        # TODO:
        # 1. 큐가 가득 찼으면 가장 오래된 것을 디큐
        # 2. 새 값 x를 인큐
        # 3. 현재 큐의 모든 값의 평균을 result에 추가
        #    (힌트: 큐를 순회하려면 (i+front)%capacity 사용, 또는 peek/deque 조합)
        ___
        ___
        # 평균 계산
        total = 0
        for i in range(q.no):
            total += q.que[(i + q.front) % q.capacity]
        result.append(round(total / q.no, 2))
    return result

stream = [10, 20, 30, 40, 50]
print(moving_average(stream, 3))
# [10.0, 15.0, 20.0, 30.0, 40.0]
# 10 / (10+20)/2 / (10+20+30)/3 / (20+30+40)/3 / (30+40+50)/3


### 9. [설명] 스택 vs 큐 vs 덱 — 언제 뭘 써?

**(a)** 지금까지 배운 3가지 자료구조를 비교하는 표를 채워봐.

| | 스택(11일차) | 큐(12일차) | 덱 |
|---|---|---|---|
| 순서 | LIFO | ? | 양쪽 자유 |
| 넣는 곳 | 꼭대기 | ? | 양쪽 |
| 빼는 곳 | 꼭대기 | ? | 양쪽 |
| 커서 개수 | 1 (ptr) | ? | ? |
| 대표 응용 | 괄호검사, undo | ? | ? |

**(b)** 다음 상황엔 각각 뭐가 맞을까?
1. 웹 브라우저 "뒤로 가기"
2. 프린터 인쇄 대기열
3. 함수 호출 스택 (재귀)
4. BFS(너비 우선 탐색)

*(여기에 답 작성)*

---
## ✅ 정답 & 해설

### R-1. 스택 vs 큐
- 큐는 인큐가 **rear(뒤)**, 디큐가 **front(앞)** — 서로 **다른 끝**에서 일어남. (스택은 둘 다 꼭대기 한 곳)
- 큐는 넣는 곳과 빼는 곳이 다르니까 **양쪽 위치를 각각** 기억해야 함 → front, rear 둘 필요. (스택은 한 곳이라 ptr 하나로 충분)
- 스택 비유: 접시 쌓기, 브라우저 뒤로가기, 실행취소(undo), 웹페이지 방문 히스토리 등 "가장 최근 것부터".

### R-2. 스택 pop 버그
**버그**: `x = self.stk[self.ptr]`를 `ptr -= 1` **전에** 함. push 후 `ptr`은 "다음 넣을 빈자리"를 가리키니까, 감소 전에 접근하면 **아직 안 넣은 빈칸(None)**을 읽어. 실행하면 30이 아니라 `None`이 나옴.
**해결**: `self.ptr -= 1`을 먼저 하고 `return self.stk[self.ptr]`. (11일차 2번 정답과 동일)

---
### 1. FIFO 추적 정답 (capacity=4)

| 연산 | front | rear | no | 큐 내용 | 반환 |
|---|---|---|---|---|---|
| enque(10) | 0 | 1 | 1 | [10] | - |
| enque(20) | 0 | 2 | 2 | [10,20] | - |
| enque(30) | 0 | 3 | 3 | [10,20,30] | - |
| deque() | 1 | 3 | 2 | [20,30] | **10** |
| enque(40) | 1 | **0** | 3 | [20,30,40] | - |
| enque(50) | 1 | 1 | 4 | [20,30,40,50] | - |

핵심: enque(40)에서 rear가 3→4가 되는데 capacity(4)에 닿아서 **0으로 순환**. enque(50)은 rear=0 자리(물리적으론 배열 맨 앞)에 저장. 논리적 순서는 여전히 20,30,40,50.

---
### 2. enque 구현 정답
```python
self.que[self.rear] = x
self.rear += 1
self.no += 1
if self.rear == self.capacity:
    self.rear = 0
```
또는 한 줄로: `self.rear = (self.rear + 1) % self.capacity` (이러면 if 불필요). 교재는 if 방식, 더 간결한 건 `%` 방식.

---
### 3. peek vs deque
- `deque`는 `front += 1`, `no -= 1`로 **커서를 움직여서** 데이터를 꺼낸 것으로 처리 → 다음엔 그 다음 원소가 맨 앞.
- `peek`은 front/rear/no를 **안 건드리고** `que[front]` 값만 반환.
- `peek` 3번 → 매번 **같은 값**(맨 앞). `deque` 3번 → 앞에서부터 **차례로 다른 값** 3개를 꺼내고 큐가 3개 줄어듦.
- 11일차 스택 peek과 완전히 같은 논리.

---
### 4. no가 필요한 이유

**(a)** 링 버퍼에서:
- **빈 상태**: 초기 front=0, rear=0 → 같음
- **가득 찬 상태**: 인큐를 capacity번 하면 rear가 한 바퀴 돌아 front와 다시 만남 → 같음

즉 front와 rear가 같아지는 상황이 **빈 것과 가득 찬 것 두 경우 모두** 발생해서, 이 둘만으론 구분 불가. `no`(실제 개수)를 봐야 `no==0`(빔) / `no==capacity`(가득)을 구별.

**(b)** `no` 없이 하는 유명한 방법: **한 칸을 항상 비워둠**. capacity가 N이면 최대 N-1개만 저장하고, "rear의 다음이 front면 가득 참"으로 판정. 이러면 빈 상태(front==rear)와 가득 상태(front==rear+1)가 구분됨. 대신 칸 하나를 낭비. 우리 교재는 `no`를 쓰는 방식(칸 낭비 없음).

---
### 5. contains & count 타입

**(a)**
```python
def count(self, value: Any) -> int:       # bool -> int
    ...
    return c
def __contains__(self, value: Any) -> bool:
    return self.count(value) > 0          # > 0 추가
```

**(b)** `count`는 "개수"를 반환하니 당연히 **int**여야 하는데 `-> bool`이라 적으면 거짓말하는 문서야. `__contains__`도 int를 그대로 반환하면 `in`은 우연히 동작하지만(bool 강제변환), 직접 호출 시 `2` 같은 값이 나와 혼란. 11일차 스택에서 지적한 것과 **완전히 같은 실수**를 큐에서 반복한 케이스 — 이런 패턴은 한 번 익히면 앞으로 클래스 짤 때마다 자동으로 걸러낼 수 있어.

---
### 6. O(n) vs O(1) 🔥

실행 결과 (N=1000): 순진한 큐 약 **499,500번** 이동 vs 링 버퍼 **2,000번** → 약 **249배** 차이.

**(a)** 순진한 큐는 디큐 1번마다 남은 원소 전부(최대 N-1개)를 앞으로 당겨. 이걸 N번 하면 대략 `(N-1) + (N-2) + ... + 1 ≈ N²/2`. 그래서 **O(N²)**. N이 10배 커지면 이동은 100배.

**(b)** 링 버퍼는 원소를 **물리적으로 안 옮겨.** 대신 **front/rear 커서(정수 하나)**만 이동시켜서 "논리적 맨 앞/맨 끝"을 바꿔. 데이터는 제자리에 있고 "어디가 시작인지"만 바뀌는 거야. 그래서 인큐·디큐 모두 **O(1)**. 배열을 원형으로 취급(`% capacity`)하는 게 이걸 가능하게 함.

핵심 통찰: **"데이터를 옮기지 말고, 관점(커서)을 옮겨라."** 이게 링 버퍼의 전부야.

---
### 7. 마지막 n개 저장

**(a)** 11번째 값(97)은 `cnt=10`일 때 저장 → `10 % 10 = 0` → 인덱스 0. 원래 인덱스 0엔 **1번째 값(15)**이 있었는데 97로 덮어써짐. 그래서 15가 버려지는 거야.

**(b)** `% n`이 하는 일이 바로 **링 버퍼 순환**이야. 인덱스가 n에 도달하면 0으로 돌아와서, 배열을 원형으로 계속 재사용해. `cnt`는 계속 증가하지만 `cnt % n`은 0~(n-1)을 뱅뱅 돌면서 가장 오래된 자리를 새 값으로 덮어써. FixedQueue의 `(i+front) % capacity`와 정확히 같은 원리. 교재 183p도 "FixedQueue의 find()에서 인덱스를 구하는 방법도 이와 같은 원리"라고 명시.

---
### 8. 이동평균 정답
```python
def moving_average(stream, window=3):
    q = FixedQueue(window)
    result = []
    for x in stream:
        if q.is_full():
            q.deque()          # 가장 오래된 것 제거
        q.enque(x)             # 새 값 추가
        total = 0
        for i in range(q.no):
            total += q.que[(i + q.front) % q.capacity]
        result.append(round(total / q.no, 2))
    return result
```
결과: `[10.0, 15.0, 20.0, 30.0, 40.0]`

**왜 큐인가**: "최근 N개"라는 슬라이딩 윈도우는 **가장 오래된 걸 버리고(디큐) 새 걸 넣는(인큐)** FIFO 패턴 그 자체. 큐가 가득 차면 front를 밀어내는 게 "윈도우가 한 칸 미끄러지는" 것과 같아. (실무에선 `collections.deque(maxlen=3)`으로 더 간단히 — 7번 last_elements와 같은 아이디어)

---
### 9. 스택 vs 큐 vs 덱

**(a)**

| | 스택 | 큐 | 덱 |
|---|---|---|---|
| 순서 | LIFO | **FIFO** | 양쪽 자유 |
| 넣는 곳 | 꼭대기 | **rear(뒤)** | 양쪽 |
| 빼는 곳 | 꼭대기 | **front(앞)** | 양쪽 |
| 커서 개수 | 1 (ptr) | **3 (front,rear,no)** | 2+ |
| 대표 응용 | 괄호검사, undo | **대기열, BFS** | 양방향 큐, 슬라이딩윈도우 |

**(b)**
1. 브라우저 뒤로가기 → **스택** (가장 최근 방문부터, LIFO)
2. 프린터 대기열 → **큐** (먼저 요청한 것부터, FIFO)
3. 함수 호출 스택 → **스택** (가장 최근 호출된 함수부터 반환, LIFO)
4. BFS → **큐** (먼저 발견한 노드부터 탐색, FIFO)

기억법: "**최근 것부터 = 스택**, **먼저 온 것부터 = 큐**, **양쪽 다 = 덱**".

---

## 📌 핵심 3줄 요약

1. **큐 = FIFO**. 인큐(rear)와 디큐(front)가 **다른 끝**에서 일어나서 커서가 3개(front, rear, no). `no`는 front==rear일 때 빈/가득을 구분하려고 필요.
2. **링 버퍼**가 오늘의 핵심 🎯 — 원소를 옮기는 대신 **커서만 순환**(`% capacity`)시켜서 디큐를 O(n)→**O(1)**로. "데이터를 옮기지 말고 관점을 옮겨라."
3. `(i + front) % capacity`는 논리→물리 인덱스 변환. `last_elements.py`의 `cnt % n`도 같은 링 버퍼 원리.

## 🗂️ 스터디 진행 가이드
- 🟢 (1~3번): 전원 필수. 1번 FIFO 추적(특히 rear 순환)은 큐 이해의 출발점
- 🟡 (4~6번): 팀 목표선.
  - **6번 O(n) vs O(1)은 오늘의 하이라이트** 🔥 — 링 버퍼가 왜 대단한지 249배 숫자로 체감
  - 4번 `no` 필요성, 5번 타입 이슈(11일차 재발)
- 🔴 (7~9번): 도전. 8번 이동평균은 큐의 실전 응용, 9번은 04장 총정리
- **금요일 코딩테스트 범위**: 🟢🟡 (1~6번)

## 🔗 04장 스택과 큐 완주! 총정리

| | 스택 (11일차) | 큐 (12일차) |
|---|---|---|
| 순서 | LIFO (후입선출) | FIFO (선입선출) |
| 넣기 | push (꼭대기) | enque (rear) |
| 빼기 | pop (꼭대기) | deque (front) |
| 커서 | ptr 1개 | front/rear/no 3개 |
| 핵심 트릭 | ptr로 꼭대기 추적 | **링 버퍼**로 O(1) 유지 |
| find 방향 | 꼭대기→바닥 | front→rear |

### 🎓 04장에서 회수된 이전 개념들
- **11일차 스택** → 큐와 LIFO/FIFO 대비, peek 동작, push/pop 순서 (R-2)
- **11일차 `__contains__` 타입** → 큐에서 똑같이 재발 (5번) 🎯
- **8~9일차 `% capacity` 순환** → 링 버퍼 커서 순환 (해시 재해시와 같은 원리!)
- **8~9일차 어노테이션** → `count -> bool`(int인데) (5번)
- **6일차 선형 검색** → `find`/`count`가 큐의 선형 검색

> **다음 진도**: 05장 이후 (재귀 / 정렬 등)로 진행될 가능성. 스택은 재귀와, 큐는 정렬·탐색과 연결돼.
